RF, Anti-CCP → impute + add _was_missing flag (their missingness was class-dependent)
Everything else (ESR, CRP, HLA-B27, ANA, Anti-Ro, Anti-La, Anti-dsDNA, Anti-Sm, C3, C4) → straight MICE, no flag needed

In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel("../data/raw/dataset.xlsx")
df.shape

(12085, 15)

In [3]:
# add missing flags because we are adding aditional column of was this originally missing needed for rf
df['RF_was_missing'] = df['RF'].isna().astype(int)
df['Anti-CCP_was_missing'] = df['Anti-CCP'].isna().astype(int)

df[['RF', 'RF_was_missing', 'Anti-CCP', 'Anti-CCP_was_missing']].head(10)

,RF,RF_was_missing,Anti-CCP,Anti-CCP_was_missing
0,34.2,0,29.9,0
1,35.5,0,28.9,0
2,21.3,0,21.3,0
3,26.0,0,39.0,0
4,38.1,0,30.8,0
5,NaN,1,37.3,0
6,22.3,0,NaN,1
7,31.8,0,38.1,0
8,33.4,0,NaN,1
9,37.4,0,25.0,0


In [4]:
df['RF_was_missing'].sum(), df['RF'].isna().sum()

(np.int64(1329), np.int64(1329))

In [5]:
#MICE needs  umeric so first convert positive/negative to 1/0
binary_cols = ['HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm']

for col in binary_cols:
    df[col] = df[col].map({'Positive': 1, 'Negative': 0})

df[binary_cols].head(10)

,HLA-B27,ANA,Anti-Ro,Anti-La,Anti-dsDNA,Anti-Sm
0,1.0,0.0,1.0,0.0,1.0,1.0
1,0.0,NaN,1.0,NaN,1.0,NaN
2,0.0,0.0,NaN,1.0,0.0,NaN
3,NaN,NaN,1.0,1.0,NaN,NaN
4,1.0,0.0,1.0,0.0,1.0,0.0
5,1.0,NaN,0.0,0.0,NaN,0.0
6,0.0,1.0,0.0,NaN,1.0,1.0
7,0.0,1.0,0.0,0.0,1.0,1.0
8,NaN,NaN,1.0,NaN,1.0,NaN
9,1.0,0.0,0.0,NaN,1.0,1.0


In [6]:
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

In [7]:
%pip install scikit-learn imbalanced-learn pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from sklearn.model_selection import train_test_split

# split BEFORE imputation so no test-set information leaks into the MICE models
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=df['Disease'], random_state=42
)

df_train = df.loc[train_idx].copy()
df_test = df.loc[test_idx].copy()

print(df_train.shape, df_test.shape)

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

mice_cols = ['ESR', 'CRP', 'RF', 'Anti-CCP', 'HLA-B27', 'ANA',
             'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm', 'C3', 'C4']

imputer = IterativeImputer(random_state=42, max_iter=50)  # 15 was too low: doesn't converge (verified converges at iter 17)
df_train[mice_cols] = imputer.fit_transform(df_train[mice_cols])
df_test[mice_cols] = imputer.transform(df_test[mice_cols])  # transform only, never fit on test

df_train[mice_cols].isna().sum(), df_test[mice_cols].isna().sum()

In [ ]:
for col in binary_cols:
    df_train[col] = df_train[col].round().clip(0, 1)
    df_test[col] = df_test[col].round().clip(0, 1)

df_train[binary_cols].describe()

In [ ]:
continuous_cols = ['ESR', 'CRP', 'RF', 'Anti-CCP', 'C3', 'C4']
df_train[continuous_cols].describe()

In [ ]:
for col in continuous_cols:
    df_train[col] = df_train[col].clip(lower=0)
    df_test[col] = df_test[col].clip(lower=0)

df_train[continuous_cols].describe()

In [ ]:
df_train[binary_cols].apply(pd.Series.value_counts)

In [ ]:
for d in (df_train, df_test):
    d['Inflammation_Score'] = d['ESR'] + d['CRP']
    d['C3_C4_Ratio'] = d['C3'] / d['C4']
    d['Autoantibody_Count'] = d[binary_cols].sum(axis=1)

df_train[['Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']].describe()

In [ ]:
df_train.groupby('Disease')[['Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']].mean().round(2)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_train['Disease_encoded'] = le.fit_transform(df_train['Disease'])
df_test['Disease_encoded'] = le.transform(df_test['Disease'])

# check the mapping so we can decode predictions back to disease names later
dict(zip(le.classes_, le.transform(le.classes_)))

In [ ]:
# Stage 1 feature set: original 14 raw features only
raw_features = ['Age', 'Gender', 'ESR', 'CRP', 'RF', 'Anti-CCP', 'HLA-B27',
                 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm', 'C3', 'C4']

# Stage 1b feature set: raw + missingness flags + engineered features
extended_features = raw_features + ['RF_was_missing', 'Anti-CCP_was_missing',
                                     'Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']

X_raw_train, X_raw_test = df_train[raw_features], df_test[raw_features]
X_ext_train, X_ext_test = df_train[extended_features], df_test[extended_features]
y_train, y_test = df_train['Disease_encoded'], df_test['Disease_encoded']

print(X_raw_train.shape, X_raw_test.shape)
print(X_ext_train.shape, X_ext_test.shape)

In production, your model receives a single new user, a new sensor ping, or a new transaction. You cannot calculate the mean or standard deviation of data you haven't received yet. Therefore, any step that uses the test set to determine how to process the training set cheats this simulation.

In [17]:
from sklearn.preprocessing import StandardScaler

continuous_cols = ['Age', 'ESR', 'CRP', 'RF', 'Anti-CCP', 'C3', 'C4']
extended_continuous_cols = continuous_cols + ['Inflammation_Score', 'C3_C4_Ratio']

extended_continuous_cols = continuous_cols + ['Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']

scaler_raw = StandardScaler()
X_raw_train_scaled = X_raw_train.copy()
X_raw_test_scaled = X_raw_test.copy()
X_raw_train_scaled[continuous_cols] = scaler_raw.fit_transform(X_raw_train[continuous_cols])
X_raw_test_scaled[continuous_cols] = scaler_raw.transform(X_raw_test[continuous_cols])

scaler_ext = StandardScaler()
X_ext_train_scaled = X_ext_train.copy()
X_ext_test_scaled = X_ext_test.copy()
X_ext_train_scaled[extended_continuous_cols] = scaler_ext.fit_transform(X_ext_train[extended_continuous_cols])
X_ext_test_scaled[extended_continuous_cols] = scaler_ext.transform(X_ext_test[extended_continuous_cols])

X_raw_train_scaled.head()

,Age,Gender,ESR,CRP,RF,Anti-CCP,HLA-B27,ANA,Anti-Ro,Anti-La,Anti-dsDNA,Anti-Sm,C3,C4
327,0.908935,0,-0.051554,-0.093818,0.728475,1.121494,0.0,1.0,0.0,1.0,1.0,0.0,0.067565,0.066177
10698,1.473599,0,1.676747,1.253853,-1.562233,-1.047476,0.0,1.0,1.0,0.0,1.0,0.0,0.678869,0.099228
6344,0.061939,1,-0.579142,-1.117981,1.276055,-0.113749,0.0,1.0,1.0,1.0,1.0,1.0,-0.211560,0.206758
4506,-1.462653,0,0.478306,1.198816,-0.786495,-0.211012,1.0,1.0,1.0,0.0,1.0,0.0,-0.601959,1.174526
5436,1.021867,0,0.760292,0.244256,0.819739,0.615726,1.0,1.0,0.0,1.0,1.0,1.0,-0.048874,0.529347


Cleaned, imputed data (MICE for most features, missing-indicator flags for RF/Anti-CCP)
Engineered features computed and held aside (Inflammation_Score, C3_C4_Ratio, Autoantibody_Count)
Two scaled feature sets with matching train/test rows: X_raw_train_scaled/X_raw_test_scaled (14 features, for Stage 1) and X_ext_train_scaled/X_ext_test_scaled (19 features, for the engineered-features comparison)
Encoded label y_train/y_test with the class mapping saved

In [18]:
import os

os.makedirs('../data/processed', exist_ok=True)

# raw feature set (Stage 1)
X_raw_train_scaled.to_csv('../data/processed/X_raw_train.csv', index=True)
X_raw_test_scaled.to_csv('../data/processed/X_raw_test.csv', index=True)

# extended feature set (Stage 1b comparison)
X_ext_train_scaled.to_csv('../data/processed/X_ext_train.csv', index=True)
X_ext_test_scaled.to_csv('../data/processed/X_ext_test.csv', index=True)

# labels (shared across both feature sets, same rows)
y_train.to_csv('../data/processed/y_train.csv', index=True)
y_test.to_csv('../data/processed/y_test.csv', index=True)

print("Saved all processed files to data/processed/")

Saved all processed files to data/processed/
